# 실전 — 실제 데이터로 검증: NSL-KDD 로 검증하기

본문(ch07~ch14)의 실습은 재현성과 안전을 위해 **합성 데이터**를 썼다. 이 노트북은 그 파이프라인을 **실제 공개 데이터셋 NSL-KDD** 에 그대로 얹어, 합성에서 배운 것이 실데이터에서도 성립하는지 검증한다.

> **인터넷 필요**: 이 노트북은 실행 시 NSL-KDD(약 5MB)를 내려받는다. 오프라인 환경에서는 동작하지 않는다(본문 노트북들은 오프라인에서도 실행됨).

핵심 관찰 세 가지를 미리 짚는다. 첫째, KDDTest+ 는 학습셋에 없는 **신규 공격 유형**을 포함하므로 정확도가 학습셋보다 크게 낮다(문헌상 RandomForest 약 77%). 이것이 실데이터의 정상적인 모습이다. 둘째, DoS·Probe 는 잘 잡히지만 **이 실험에서는 R2L·U2R의 재현율이 낮다**. 15장은 별도 HTTP 페이로드 분류 과제이며 이 결과의 직접적인 해결책은 아니다. 셋째, 같은 전처리(ch09)·모델(ch14) 코드가 데이터만 바꿔 그대로 동작한다.

## 데이터셋 이야기 — 어떻게 수집됐나 (NSL-KDD)

뿌리는 **1998년 미국 DARPA 침입 탐지 평가**다. MIT 링컨연구소가 공군 LAN 을 모사한 네트워크에 수 주간 정상 트래픽과 실제 공격(DoS·스캔·원격침투·권한상승)을 주입해 tcpdump 로 수집했고, KDD Cup 1999 가 이를 41개 feature 로 가공했다. **NSL-KDD(2009, UNB)** 는 그 데이터의 **중복 레코드를 제거**해 모델이 빈발 패턴에 편향되지 않도록 정제한 판이다.

**시나리오**: 군용 네트워크에 다양한 공격이 섞여 들어올 때 41개 통계 feature 로 정상/공격을 가려낼 수 있는가.

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# 필요 패키지 (Colab 등에서 최초 1회만 주석 해제)
# !pip install pandas scikit-learn matplotlib

# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

## 1. 실데이터 내려받기

NSL-KDD 는 KDDCup99 의 중복을 제거한 침입 탐지 표준 데이터셋이다. 공개 미러에서 `KDDTrain+`·`KDDTest+` 를 받는다(다운로드 실패 시 대체 URL 로 재시도).

In [ ]:
import urllib.request, ssl, os

MIRRORS = [
    'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/',
    'https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/',
]
def fetch(fname):
    """로컬 캐시가 있으면 재사용, 없으면 미러에서 다운로드."""
    if os.path.exists(fname):
        return fname
    last = None
    for base in MIRRORS:
        try:
            url = base + fname.replace('+', '%2B')
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=60, context=ssl.create_default_context()) as r:
                open(fname, 'wb').write(r.read())
            return fname
        except Exception as e:
            last = e
    raise RuntimeError(f'다운로드 실패({fname}): {last}')

train_path = fetch('KDDTrain+.txt')
test_path  = fetch('KDDTest+.txt')
print('다운로드 완료:', train_path, test_path)

## 2. 로드와 스키마

NSL-KDD 는 헤더가 없다. 41개 feature + `label` + `difficulty` 의 43개 컬럼명을 직접 부여한다. 본문 ch08 에서 본 그 스키마다.

In [ ]:
COLS = ['duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
        'wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised',
        'root_shell','su_attempted','num_root','num_file_creations','num_shells',
        'num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count',
        'srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
        'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count',
        'dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
        'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
        'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate',
        'label','difficulty']

train = pd.read_csv(train_path, names=COLS)
test  = pd.read_csv(test_path,  names=COLS)
print('train:', train.shape, ' test:', test.shape)
train[['protocol_type','service','flag','src_bytes','count','label']].head()

## 3. 라벨을 5개 카테고리로

NSL-KDD 의 세부 공격명 40여 종을 표준 5분류(normal·DoS·Probe·R2L·U2R)로 묶는다. 14.9절에서 정리한 공격 카테고리 체계와 같은 관점이다. 상수 컬럼 `num_outbound_cmds`(항상 0)와 학습에 불필요한 `difficulty` 는 제거한다.

In [ ]:
ATTACK_MAP = {
    **{a: 'DoS'   for a in ['back','land','neptune','pod','smurf','teardrop','mailbomb',
                            'apache2','processtable','udpstorm','worm']},
    **{a: 'Probe' for a in ['satan','ipsweep','nmap','portsweep','mscan','saint']},
    **{a: 'R2L'   for a in ['guess_passwd','ftp_write','imap','phf','multihop','warezmaster',
                            'warezclient','spy','xlock','xsnoop','snmpguess','snmpgetattack',
                            'httptunnel','sendmail','named']},
    **{a: 'U2R'   for a in ['buffer_overflow','loadmodule','rootkit','perl','sqlattack',
                            'xterm','ps']},
}
for df in (train, test):
    df['category'] = df['label'].map(lambda x: 'normal' if x == 'normal' else ATTACK_MAP.get(x, 'unknown'))
    df.drop(columns=['difficulty', 'num_outbound_cmds'], inplace=True)

print('KDDTest+ 카테고리 분포:')
print(test['category'].value_counts())

## 4. 전처리 파이프라인 (ch09 과 동일)

범주형 3개(`protocol_type`·`service`·`flag`)는 One-Hot, 수치형은 표준화한다. `handle_unknown='ignore'` 로 테스트셋에만 등장하는 service 값을 안전하게 처리한다 — 실데이터에서 반드시 마주치는 함정이다.

In [ ]:
cat_cols = ['protocol_type', 'service', 'flag']
num_cols = [c for c in train.columns if c not in cat_cols + ['label', 'category']]

pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
print(f'수치형 {len(num_cols)}개, 범주형 {len(cat_cols)}개')

## 5. 이진 탐지 (normal vs attack)

ch14 의 RandomForest 를 그대로 쓴다. 학습은 KDDTrain+, 평가는 **신규 공격이 섞인 KDDTest+** 로 한다.

In [ ]:
# ── 실습 과제 ──
# 목표: 5. 이진 탐지 (normal vs attack) — ch14 의 RandomForest 를 그대로 쓴다. 학습은 KDDTrain+, 평가는 **신규 공격이 섞인 KDDTest+** 로 한다.
# 만들 결과: y_train, y_test, clf, proba, pred
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
y_train = ...
y_test = ...

clf = ...
    ...
    ('rf', RandomForestClassifier(n_estimators = ...
...

proba = ...
pred = ...

print(classification_report(y_test, pred, target_names = ...
...
```
</details>


In [ ]:
# 혼동 행렬 시각화
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['normal', 'attack'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['normal', 'attack'])
ax.set_xlabel('예측'); ax.set_ylabel('실제'); ax.set_title('KDDTest+ 이진 탐지')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout(); plt.show()

## 6. 5개 카테고리 분류 — R2L·U2R 의 벽

이제 5분류로 학습한다. **DoS·Probe 는 잘 잡히지만 R2L·U2R 의 recall 은 바닥에 가깝다.** 이 결과만으로 원인을 페이로드 정보의 부족으로 단정할 수는 없다. 14장의 클래스별 평가 관점으로 표본 수·특성·새 공격 유형의 영향을 검토한다. 15장의 HTTP 페이로드 분류와는 데이터와 목표가 다르다.

In [ ]:
# ── 실습 과제 ──
# 목표: 6. 5개 카테고리 분류 — R2L·U2R 의 벽 — 이제 5분류로 학습한다. **DoS·Probe 는 잘 잡히지만 R2L·U2R 의 recall 은 바닥에 가깝다.** 이 두 유형은 정상 트래픽과 통계적으로 거의 구분되지 않고, 공격 신호가 **페이로드 내용**에 있기 때문이다. 표 
# 만들 결과: clf5, pred5
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
clf5 = ...
    ...
    ('rf', RandomForestClassifier(n_estimators = ...
...

pred5 = ...
print(classification_report(test['category'], pred5, digits = ...
```
</details>


## 7. 정리 — 합성에서 실데이터로

| 관찰 | 합성 데이터(본문) | 실제 NSL-KDD(KDDTest+) |
|---|---|---|
| 이진 탐지 정확도 | 0.95+ | 약 0.77 |
| DoS / Probe recall | 높음 | 0.8 / 0.6 대 |
| **R2L / U2R recall** | 낮게 설정 | **0.0~0.1 (거의 못 잡음)** |

세 가지 교훈이 실데이터에서 그대로 확인된다.

1. **테스트 정확도가 학습보다 크게 낮다.** KDDTest+ 는 학습셋에 없는 신규 공격을 포함한다 — 이것이 드리프트(ch16)이자, 정적 모델의 근본 한계다.
2. **이 실험에서는 R2L·U2R의 재현율이 낮다.** 14장의 클래스별 평가와 16장의 오류 분석으로 원인을 조사한다. 15장(NLP)·Part IV(LLM)의 예제를 적용하면 이 문제가 해결된다는 뜻은 아니다.
3. **같은 전처리·모델 코드가 데이터만 바꿔 그대로 돌아간다.** 본문에서 합성으로 익힌 파이프라인은 실데이터 이식이 목적이었고, 이 노트북이 그 다리를 완성한다.

> 더 나아가려면: CICIDS2017 의 하루치 CSV(플로우 기반, 79 feature)로 같은 실험을 반복해 보라. 데이터 규모와 feature 성격이 달라도 파이프라인 골격은 동일하다.